# Task4: Notebook نهاية الاسبوع

## 1. Data Understanding

الهدف من المشروع هو تحليل بيانات طلبات القروض ومعرفة العوامل التي قد تؤثر على قبول أو رفض طلب القرض.

- Target: Loan_Status
- Features: باقي الأعمدة التي تحتوي على معلومات المتقدم وبيانات القرض.
- نوع المشكلة: تصنيف (Classification)، لأننا نتوقع قبول الطلب أو رفضه.

In [14]:
import pandas as pd

df = pd.read_csv("loan_clean.csv")

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1.0,Urban,1
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,0
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,1
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,1
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,1


**(EDA)**

في هذا القسم نستكشف البيانات من خلال معرفة حجمها، وأنواع الأعمدة، والقيم الناقصة والتكرار، وبعض الإحصائيات.

In [15]:
print("Data Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

print("\nStatistics:")
df.describe()

Data Shape:
(614, 13)

Data Types:
Loan_ID               object
Gender                object
Married               object
Dependents             int64
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status            int64
dtype: object

Missing Values:
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Duplicates:
0

Statistics:


,Dependents,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.744300,5403.459283,1621.245798,145.752443,342.410423,0.855049,0.687296
std,1.009623,6109.041673,2926.248369,84.107233,64.428629,0.352339,0.463973
min,0.000000,150.000000,0.000000,9.000000,12.000000,0.000000,0.000000
25%,0.000000,2877.500000,0.000000,100.250000,360.000000,1.000000,0.000000
50%,0.000000,3812.500000,1188.500000,128.000000,360.000000,1.000000,1.000000
75%,1.000000,5795.000000,2297.250000,164.750000,360.000000,1.000000,1.000000
max,3.000000,81000.000000,41667.000000,700.000000,480.000000,1.000000,1.000000


**ملاحظات على البيانات**

- حجم البيانات 614 صفًا و13 عمودًا.
- البيانات تحتوي على أعمدة رقمية وتصنيفية.
- لا توجد قيم ناقصة في البيانات.
- لا توجد صفوف مكررة.
- متوسط دخل المتقدم حوالي 5403.
- متوسط مبلغ القرض حوالي 145.
- نسبة الموافقة على القروض حوالي 68.7%.
- أغلب المتقدمين لديهم سجل ائتماني جيد، بمتوسط 0.85.

In [16]:
#Feature Engineering

# إنشاء إجمالي الدخل
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

# معرفة هل يوجد شريك في الطلب
df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)

# حساب نسبة مبلغ القرض إلى الدخل
df["Loan_to_Income"] = (df["LoanAmount"] * 1000) / df["Total_Income"]

# حساب القسط الشهري
df["Monthly_Payment"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]

# حساب نسبة القسط إلى الدخل
df["Payment_to_Income"] = df["Monthly_Payment"] / df["Total_Income"]

**Feature Selection**

بعد تحليل الأعمدة ومقارنة نتائجها، اتخذت القرارات التالية:

| العمود | القرار | السبب |
|---|---|---|
| Loan_ID | حذف | عمود تعريفي |
| Gender | حذف | مراعاة العدالة، رغم أن الفرق في الموافقة بسيط (0.021) |
| Married | إبقاء | فرق الموافقة 0.089 |
| Dependents | إبقاء | فرق الموافقة 0.105 |
| Education | إبقاء | فرق الموافقة 0.096 |
| Self_Employed | حذف | فرق الموافقة شبه معدوم (0.005) |
| ApplicantIncome | حذف | ارتباطه بالهدف شبه معدوم، ومرتبط بقوة مع Total_Income |
| CoapplicantIncome | حذف | ارتباطه بالهدف ضعيف، ويدخل ضمن Total_Income |
| LoanAmount | حذف | ارتباطه بالهدف ضعيف (-0.033) |
| Loan_Amount_Term | حذف | ارتباطه بالهدف ضعيف (-0.023) |
| Credit_History | إبقاء | ارتباطه بالهدف 0.541 |
| Property_Area | إبقاء | فرق الموافقة 0.154 |
| Total_Income | إبقاء | يمثل مجموع دخل المتقدم والشريك |
| Has_Coapplicant | إبقاء | فرق الموافقة 0.070 |
| Loan_to_Income | إبقاء | ارتباطه بالهدف -0.086 |
| Monthly_Payment | حذف | ارتباطه بالهدف شبه معدوم |
| Payment_to_Income | إبقاء | توجد اختلافات في نسب الموافقة بين فئاته |
| Payment_Category | حذف | يمثل فئات لعمود Payment_to_Income |
| Income_Category | حذف | فرق الموافقة بسيط (0.011) |

كان الارتباط بين ApplicantIncome وTotal_Income حوالي 0.893، لذلك اخترت الاحتفاظ بـ Total_Income لأنه يشمل دخل المتقدم والشريك.

وبالنسبة إلى Gender، قررت حذفه مراعاةً للعدالة وعدم استخدام الجنس كعامل في قرار قبول القرض.

**Save Model-Ready Data**

بعد اختيار الأعمدة المناسبة، حفظت البيانات النهائية في ملف `model_ready.csv`

ليكون جاهزًا للاستخدام في مرحلة
بناء النموذج.

In [17]:
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)

df["Loan_to_Income"] = (df["LoanAmount"] * 1000) / df["Total_Income"]

df["Monthly_Payment"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]

df["Payment_to_Income"] = df["Monthly_Payment"] / df["Total_Income"]

In [18]:
selected_columns = [
    "Married", "Dependents", "Education", "Credit_History",
    "Property_Area", "Total_Income", "Has_Coapplicant",
    "Loan_to_Income", "Payment_to_Income", "Loan_Status"
]

model_ready = df[selected_columns]

model_ready.to_csv("model_ready.csv", index=False)

print(model_ready.shape)

(614, 10)


## Final Report

### أهم الاستنتاجات من البيانات

1. تحتوي البيانات على 614 طلب قرض.
2. نسبة الموافقة على القروض كانت حوالي 68.7%.
3. متوسط دخل المتقدم كان حوالي 5403.
4. كان الارتباط بين Credit_History وLoan_Status حوالي 0.54، وهو أعلى ارتباط بين الأعمدة الرقمية.
5. كان الارتباط بين ApplicantIncome وTotal_Income حوالي 0.89، مما يدل على وجود تشابه كبير بينهما.

### مقارنة التوقعات بالنتائج

بعض توقعاتي كانت صحيحة، لكن النتائج أظهرت أن Credit_History له تأثير أوضح من الدخل وحده. كذلك لاحظت أن وجود شريك في الطلب ارتبط بنسبة موافقة أعلى.

### توصية للبنك

أنصح البنك بالاهتمام بالسجل الائتماني عند دراسة طلبات القروض، لأنه أظهر ارتباطًا واضحًا بالموافقة.

### الرأي حول Gender

أفضل عدم استخدام Gender في اتخاذ قرار الموافقة على القروض، وذلك مراعاةً للعدالة وعدم التمييز بين المتقدمين.